# Сверка Excel (март) vs ODS по договорам и ИНН

Тетрадка делает сверку мартовского Excel-отчета с ODS-источником:
- по `agr_id` (договоры),
- по `ИНН` (с нормализацией и учетом выпадающих ведущих нулей).

Что получите на выходе:
- пересечение договоров,
- список договоров из Excel, не найденных в ODS,
- проверку несовпадений ИНН,
- отладочную проверку конкретного `agr_id`.

In [ ]:
import pandas as pd

# --- параметры запуска ---
excel_path = "/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx"
excel_header = 0

date_from = "2026-03-01"
date_to = "2026-03-31"

agr_col = "ID договора"
inn_col = "ИНН"

# точечная проверка проблемного договора (можно поменять)
check_agr_id = "636990046089"

In [ ]:
from rail_connectors.connection import connect

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)

imp._init_connection()
print('Impala connection initialized')

In [ ]:
# 1) Загрузка Excel + нормализация ключей

def normalize_agr(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\\.0$", "", regex=True)  # Excel иногда читает ID как float
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )


def normalize_inn(series: pd.Series) -> pd.Series:
    # Важно: ИНН нормализуем как строку цифр и сохраняем ведущие нули.
    s = series.astype(str).str.strip().replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    s = s.str.replace(r"\\.0$", "", regex=True)      # если пришло из float
    s = s.str.replace(r"\\D", "", regex=True)        # оставляем только цифры
    s = s.where(s.str.len().isin([10, 12]), pd.NA)      # валидная длина ИНН
    return s


df = pd.read_excel(excel_path, header=excel_header)

required_cols = [agr_col, inn_col, "Дата регистрации договора", "Дата закрытия договора"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"В Excel не найдены колонки: {missing}. Доступные: {list(df.columns)}")

df = df.copy()
df["agr_id_norm"] = normalize_agr(df[agr_col])
df["inn_norm"] = normalize_inn(df[inn_col])

excel_agr = df["agr_id_norm"].dropna().drop_duplicates()
excel_inn = df["inn_norm"].dropna().drop_duplicates()

print("excel_rows =", len(df))
print("excel_unique_agr =", len(excel_agr))
print("excel_unique_inn =", len(excel_inn))
display(df[[agr_col, "agr_id_norm", inn_col, "inn_norm"]].head(10))

In [ ]:
# 2) Выгрузка из ODS за март (логика техлида)
with imp:
    acq = imp.fetch(f"""
        select
            cast(m.id as string) as agr_id,
            cast(m.c_cl_org as string) as cft_id,
            cast(cl.c_inn as string) as c_inn
        from ods.scd1_z_r2_ip_merchants m
        left join ods.scd1_z_client cl
            on cl.id = m.c_cl_org
        where
            m.c_date_begin <= '{date_to}'
            and (
                m.c_date_close > '{date_to}'
                or m.c_date_close is null
                or m.c_date_close between '{date_from}' and '{date_to}'
            )
    """)

acq = acq.copy()
acq["agr_id_norm"] = normalize_agr(acq["agr_id"])
acq["inn_norm"] = normalize_inn(acq["c_inn"])

ods_agr = acq["agr_id_norm"].dropna().drop_duplicates()
ods_inn = acq["inn_norm"].dropna().drop_duplicates()

print("ods_rows =", len(acq))
print("ods_unique_agr =", len(ods_agr))
print("ods_unique_inn =", len(ods_inn))
display(acq[["agr_id", "agr_id_norm", "c_inn", "inn_norm"]].head(10))

In [ ]:
# 3) Сравнение множеств по договорам и ИНН
set_excel_agr = set(excel_agr.tolist())
set_ods_agr = set(ods_agr.tolist())

agr_intersect = set_excel_agr & set_ods_agr
agr_only_excel = set_excel_agr - set_ods_agr
agr_only_ods = set_ods_agr - set_excel_agr

set_excel_inn = set(excel_inn.tolist())
set_ods_inn = set(ods_inn.tolist())

inn_intersect = set_excel_inn & set_ods_inn
inn_only_excel = set_excel_inn - set_ods_inn
inn_only_ods = set_ods_inn - set_excel_inn

summary = pd.DataFrame([
    {"metric": "excel_unique_agr", "value": len(set_excel_agr)},
    {"metric": "ods_unique_agr", "value": len(set_ods_agr)},
    {"metric": "agr_intersect_cnt", "value": len(agr_intersect)},
    {"metric": "agr_only_excel_cnt", "value": len(agr_only_excel)},
    {"metric": "agr_only_ods_cnt", "value": len(agr_only_ods)},
    {"metric": "excel_unique_inn", "value": len(set_excel_inn)},
    {"metric": "ods_unique_inn", "value": len(set_ods_inn)},
    {"metric": "inn_intersect_cnt", "value": len(inn_intersect)},
    {"metric": "inn_only_excel_cnt", "value": len(inn_only_excel)},
    {"metric": "inn_only_ods_cnt", "value": len(inn_only_ods)},
])

display(summary)

In [ ]:
# 4) Расхождения: Excel договоры, не найденные в ODS
not_found_agr_df = (
    df[[agr_col, inn_col, "Дата регистрации договора", "Дата закрытия договора", "agr_id_norm", "inn_norm"]]
    .merge(
        acq[["agr_id", "cft_id", "c_inn", "agr_id_norm", "inn_norm"]],
        on="agr_id_norm",
        how="left",
        suffixes=("_excel", "_ods")
    )
    .query("agr_id.isna()")
    .drop(columns=["agr_id_norm"])
)

print("Договоров из Excel, не найденных в ODS:", len(not_found_agr_df))
display(not_found_agr_df.head(100))

# Отдельно: ИНН из Excel, которых нет в ODS
inn_only_excel_df = pd.DataFrame({"inn_norm": sorted(list(inn_only_excel))})
print("ИНН только в Excel:", len(inn_only_excel_df))
display(inn_only_excel_df.head(100))

In [ ]:
# 5) Проверка несоответствия ИНН по совпавшим договорам
matched = (
    df[[agr_col, inn_col, "agr_id_norm", "inn_norm"]]
    .merge(
        acq[["agr_id", "c_inn", "agr_id_norm", "inn_norm"]],
        on="agr_id_norm",
        how="inner",
        suffixes=("_excel", "_ods")
    )
    .rename(columns={
        "inn_norm_excel": "inn_excel_norm",
        "inn_norm_ods": "inn_ods_norm"
    })
)

inn_mismatch = matched[
    matched["inn_excel_norm"].notna() &
    matched["inn_ods_norm"].notna() &
    (matched["inn_excel_norm"] != matched["inn_ods_norm"])
].copy()

print("Совпавших договоров (inner join):", len(matched))
print("Из них с несовпадающим ИНН:", len(inn_mismatch))
display(inn_mismatch[[agr_col, "agr_id", inn_col, "c_inn", "inn_excel_norm", "inn_ods_norm"]].head(100))

In [ ]:
# 6) Точечная проверка одного договора в ODS
with imp:
    check_row = imp.fetch(f"""
        select *
        from ods.scd1_z_r2_ip_merchants
        where cast(id as string) = '{check_agr_id}'
    """)

print("rows for check_agr_id in ODS:", len(check_row))
display(check_row)


In [ ]:
# 7) Какие договоры есть только в ODS (не в Excel)
agr_only_ods_df = pd.DataFrame({"agr_id_norm": sorted(list(agr_only_ods))})
print("Договоров только в ODS:", len(agr_only_ods_df))
display(agr_only_ods_df.head(100))


In [ ]:
# 8) Диагностика дублей ключей
excel_dup_agr = (
    df[df["agr_id_norm"].notna()]
    .groupby("agr_id_norm", as_index=False)
    .size()
    .query("size > 1")
    .sort_values("size", ascending=False)
)

excel_dup_inn = (
    df[df["inn_norm"].notna()]
    .groupby("inn_norm", as_index=False)
    .size()
    .query("size > 1")
    .sort_values("size", ascending=False)
)

print("Дублей agr_id в Excel:", len(excel_dup_agr))
display(excel_dup_agr.head(20))

print("Дублей ИНН в Excel:", len(excel_dup_inn))
display(excel_dup_inn.head(20))


In [ ]:
# 9) Итоговые выборки для выгрузки (если нужно сохранить результаты)
result_not_found = not_found_agr_df.copy()
result_inn_mismatch = inn_mismatch.copy()

print("Готово. Доступны DataFrame:")
print("- summary")
print("- result_not_found")
print("- result_inn_mismatch")
